In [ ]:
# 📦 Inference & Demo (copy this whole cell into a new notebook and run)

import json, pathlib
import pandas as pd

# Optional imports (only needed if model artifacts exist)
try:
    import joblib
    SK_OK = True
except Exception:
    SK_OK = False

# ---------------------------
# Paths and artifact loading
# ---------------------------
BASE = pathlib.Path.cwd()
DATA = BASE / "data"
ART = BASE / "artifacts"

def load_json_safe(path, default=None):
    try:
        return json.loads(path.read_text())
    except Exception:
        return default

legacy_map = load_json_safe(DATA / "ncbi_protein_legacy_map.json", default={})
seed_classification = load_json_safe(ART / "seed_classification.json", default={})
eti_list = set(load_json_safe(ART / "eti_list.json", default=[]))
ivacaftor_list = set(load_json_safe(ART / "ivacaftor_list.json", default=[]))

feature_order = load_json_safe(ART / "feature_order.json", default=["pathogenic_count","mod_group","age","sex_male"])

model = None
if SK_OK and (ART / "severity_model.joblib").exists():
    model = joblib.load(ART / "severity_model.joblib")
    MODEL_READY = True
else:
    MODEL_READY = False

print({
    "legacy_map": len(legacy_map),
    "seed_classification": len(seed_classification),
    "eti_list": len(eti_list),
    "ivacaftor_list": len(ivacaftor_list),
    "model_ready": MODEL_READY
})

# ---------------------------
# Core utilities (standalone)
# ---------------------------
def standardize_variant(v):
    if v is None:
        return None
    return legacy_map.get(v, v)

def get_pathogenicity(v):
    v = standardize_variant(v)
    if not v:
        return "uncertain"
    if v in seed_classification:
        return seed_classification[v]
    if v in ivacaftor_list:
        return "responsive_ivacaftor"
    if v in eti_list:
        return "responsive_ETI"
    return "uncertain"

def is_pathogenic_like(c):
    return c in ("pathogenic", "responsive_ivacaftor", "responsive_ETI")

def pathogenic_count(v1, v2):
    return sum(is_pathogenic_like(get_pathogenicity(v)) for v in (v1, v2))

def get_modulator_group(v1, v2):
    v1s, v2s = standardize_variant(v1), standardize_variant(v2)
    vs = {v1s, v2s}
    if "F508del" in vs and len(vs) == 1:
        return 1  # F508del homozygous
    if "F508del" in vs:
        return 2  # F508del heterozygous
    if (v1s in eti_list) or (v2s in eti_list):
        return 3  # ETI-responsive present
    if (v1s in ivacaftor_list) or (v2s in ivacaftor_list):
        return 4  # ivacaftor-responsive present
    return 5      # no known modulator list match

def classify_cf(v1, v2):
    c1, c2 = get_pathogenicity(v1), get_pathogenicity(v2)
    if is_pathogenic_like(c1) and is_pathogenic_like(c2):
        return "CF positive"
    if (is_pathogenic_like(c1) and c2 == "uncertain") or (is_pathogenic_like(c2) and c1 == "uncertain"):
        return "risk uncertain"
    return "CF negative / carrier"

def make_features(variant1, variant2, age=None, sex=None):
    return {
        "pathogenic_count": pathogenic_count(variant1, variant2),
        "mod_group": get_modulator_group(variant1, variant2),
        "age": -1 if age is None else age,
        "sex_male": 1 if str(sex).upper().startswith("M") else 0
    }

# ---------------------------
# Public prediction function
# ---------------------------
def predict_cf(variant1, variant2, age=None, sex=None):
    """
    Inputs:
      - variant1, variant2: protein-level labels (e.g., 'F508del', 'G551D')
      - age: optional int
      - sex: 'M' or 'F' (optional)
    Returns:
      dict with cf_risk, modulator_group, severity_index (if model available), and expanded context.
    """
    v1s, v2s = standardize_variant(variant1), standardize_variant(variant2)
    cf_risk = classify_cf(v1s, v2s)
    feats = make_features(v1s, v2s, age, sex)

    severity_index = None
    if MODEL_READY:
        X = pd.DataFrame([[feats[k] for k in feature_order]], columns=feature_order)
        severity_index = float(model.predict(X)[0])

    return {
        "input": {
            "variant1_raw": variant1, "variant2_raw": variant2,
            "variant1_std": v1s, "variant2_std": v2s,
            "age": age, "sex": sex
        },
        "cf_risk": cf_risk,
        "modulator_group": feats["mod_group"],
        "severity_index": None if severity_index is None else round(severity_index, 3),
        "details": {
            "pathogenicity_v1": get_pathogenicity(v1s),
            "pathogenicity_v2": get_pathogenicity(v2s),
            "features": feats,
            "feature_order": feature_order
        }
    }

# ---------------------------
# Quick tests
# ---------------------------
tests = [
    {"variant1":"F508del","variant2":"F508del","age":14,"sex":"F"},
    {"variant1":"F508del","variant2":"G542X","age":12,"sex":"M"},
    {"variant1":"F508del","variant2":"G551D","age":22,"sex":"F"},
    {"variant1":"R117H","variant2":"unknown","age":10,"sex":"M"},
]

print("\nQuick tests:")
for t in tests:
    print(t, "=>", predict_cf(**t))

# ---------------------------
# Optional: Minimal Flask API
# ---------------------------
# To run an API in this notebook, uncomment lines below and execute the cell.
# Then POST JSON to: http://127.0.0.1:5000/predict
#
# Example:
# curl -X POST http://127.0.0.1:5000/predict \
#      -H "Content-Type: application/json" \
#      -d '{"variant1":"F508del","variant2":"G551D","age":22,"sex":"F"}'

# try:
#     from flask import Flask, request, jsonify
#     app = Flask(__name__)
# 
#     @app.route("/predict", methods=["POST"])
#     def api_predict():
#         data = request.get_json(force=True)
#         out = predict_cf(
#             data.get("variant1"),
#             data.get("variant2"),
#             data.get("age"),
#             data.get("sex")
#         )
#         return jsonify(out)
# 
#     print("\nFlask API ready. To launch, run: app.run(host='127.0.0.1', port=5000, debug=False)")
# except Exception as e:
#     print("[info] Flask not available in this environment:", e)


{'legacy_map': 710, 'seed_classification': 6, 'eti_list': 13, 'ivacaftor_list': 12, 'model_ready': True}

Quick tests:
{'variant1': 'F508del', 'variant2': 'F508del', 'age': 14, 'sex': 'F'} => {'input': {'variant1_raw': 'F508del', 'variant2_raw': 'F508del', 'variant1_std': 'F508del', 'variant2_std': 'F508del', 'age': 14, 'sex': 'F'}, 'cf_risk': 'CF positive', 'modulator_group': 1, 'severity_index': 0.181, 'details': {'pathogenicity_v1': 'pathogenic', 'pathogenicity_v2': 'pathogenic', 'features': {'pathogenic_count': 2, 'mod_group': 1, 'age': 14, 'sex_male': 0}, 'feature_order': ['pathogenic_count', 'mod_group', 'age', 'sex_male']}}
{'variant1': 'F508del', 'variant2': 'G542X', 'age': 12, 'sex': 'M'} => {'input': {'variant1_raw': 'F508del', 'variant2_raw': 'G542X', 'variant1_std': 'F508del', 'variant2_std': 'G542X', 'age': 12, 'sex': 'M'}, 'cf_risk': 'CF positive', 'modulator_group': 2, 'severity_index': 0.216, 'details': {'pathogenicity_v1': 'pathogenic', 'pathogenicity_v2': 'pathogenic'